# Практикум 07 · Геометрия маркера и ALVAR

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

Стандартный **Run All** использует учебные изображения или синтетические данные. Подготовьте Python с OpenCV и NumPy по стартовой инструкции.


# 9.7. ARTag/ALVAR: обнаружение метки и её положение

[Назад: 9.6](../lessons/06_ar.md) · [Оглавление](../README.md) · [Далее: 9.8](../lessons/08_ros_image.md)

**Результат:** обнаружение настоящего маркера ALVAR в заводской ROS1-среде ROSMASTER, подтверждённый ID и положение в известной системе координат. Подготовительную геометрию можно выполнить на ноутбуке. Ориентир: 60–90 минут после калибровки камеры.

## 1. Разберитесь, какую метку ищет программа

В главе Yahboom 9.7 используется пакет `ar_track_alvar`. Он работает с собственными ARTag/ALVAR-маркерами. При изготовлении метки используйте генератор из этого пакета. QR-код и метка OpenCV ArUco принадлежат другим форматам: их изображения не являются заменой ALVAR.

Детектор решает несколько разных задач: находит границу, читает ID, связывает наблюдаемые углы с известной геометрией и вычисляет положение относительно камеры. Для последних вычислений нужны параметры камеры и физический размер метки.

## 2. Проверьте заводскую среду

На ROSMASTER используйте терминал той ROS1-среды или контейнера, где доступен заводской workspace. Общая подготовка описана в [инструкции настройки](../docs/SETUP_ROBOT.md). Запустите:

```bash
python3 -m rosmaster_cv.lesson07_markers --diagnose
rospack find ar_track_alvar
rospack find yahboomcar_visual
```

Учебная диагностика сохраняет `outputs/07/diagnostic.json`. Она проверяет наличие пакета, не открывая камеру. Если `rospack` не найден, вернитесь к правильному терминалу и `setup.bash`. На ноутбуке отсутствие этого пакета ожидаемо: там сначала выполните пункт 5.

## 3. Изготовьте маркер

В отдельной рабочей папке запустите заводской генератор:

```bash
mkdir -p data/alvar
cd data/alvar
rosrun ar_track_alvar createMarker 11
```

Команда создаёт изображение маркера ID 11. Откройте полученный файл, распечатайте его, закрепите плоско и измерьте сторону чёрного квадрата. В заводском примере размер `marker_size` равен **5.0 сантиметра**. Белое поле вокруг изображения в этот размер не включается. Настройка должна соответствовать вашему реальному отпечатку: ошибка масштаба переносится в вычисленное расстояние.

Вернитесь в корень курса. Найдите launch-файл:

```bash
rospack find yahboomcar_visual
```

В папке пакета откройте `launch/ar_track.launch` текстовым редактором. Проверьте `marker_size`, `cam_image_topic`, `cam_info_topic`, `output_frame`. В опубликованном примере используются `/usb_cam/image_raw`, `/usb_cam/camera_info` и `/usb_cam`. Значения на другой комплектации камеры могут отличаться. Сверьте их с `rostopic list` и своей калибровкой.

## 4. Запустите обнаружение

```bash
roslaunch yahboomcar_visual ar_track.launch open_rviz:=true
```

В проверенном заводском launch уже подключается `usb_cam-test.launch`, поэтому отдельный повторный запуск USB-камеры здесь не нужен. Покажите камере целую метку при равномерном освещении. В RViz выберите корректный Fixed Frame, соответствующий фактическим сообщениям и TF.

В другом терминале выполните:

```bash
rostopic list
```

Найдите опубликованный топик обнаружений ALVAR. Для выбранного имени используйте `rostopic type ИМЯ_ТОПИКА`, затем `rostopic echo -n 1 ИМЯ_ТОПИКА`. В отчёт перепишите ID, `header.frame_id` и координаты. Название топика сначала наблюдают в текущей системе; его не следует угадывать по чужому скриншоту.

Переместите метку ближе и дальше, затем поверните её. Запишите, когда обнаружение пропадает и как меняются координаты. По завершении остановите launch сочетанием Ctrl+C.

## 5. Подготовительная практика без ROS и камеры

Используйте картинку из урока 9.6:

```bash
python -m rosmaster_cv.lesson07_markers --input outputs/06/checkerboard.png --corners 100 100 500 100 500 350 100 350
```

Получатся `geometry.png` и `geometry.json`. Четыре точки обходят контур по периметру; программа вычисляет площадь, длины рёбер и центр. Это упражнение на геометрию вручную заданной области. Поле `decoded_marker_id` остаётся `null`: распознавание ALVAR на этом этапе не выполняется. Подайте точки в пересекающемся порядке и объясните сообщение об ошибке.

## Задание и проверка

Сдайте фото напечатанной метки с линейкой, параметры её размера, скриншот RViz и одно настоящее сообщение с ID. Добавьте три измерения при разных положениях. Объясните разницу между пикселями изображения и координатами позы, значение `frame_id` и влияние неверного размера метки. При отсутствии робота сдаётся подготовительная геометрия с отметкой «ALVAR на оборудовании ещё не проверен»; аппаратную часть выполняют следующим доступным занятием.

**Источники:** [Yahboom 9.7 — ARTag/ALVAR](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8148/), [репозиторий ar_track_alvar](https://github.com/ros-perception/ar_track_alvar).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


In [ ]:
run_module("lesson07_markers", "--diagnose")


Следующая ячейка показывает преобразование по **четырём явно заданным углам**. Она не обнаруживает ALVAR или ArUco и не выдаёт эти углы за результат детектора. ALVAR запускается отдельно по инструкции главы.


In [ ]:
run_module("lesson06_ar", "--synthetic")
run_module("lesson07_markers", "--input", "outputs/06/checkerboard.png",
           "--corners", "100", "100", "500", "100", "500", "350", "100", "350")


In [ ]:
def show_results(relative_directory, limit=4):
    directory = ROOT / relative_directory
    paths = sorted(directory.glob("*.png"))[:limit]
    if not paths:
        print("Изображений пока нет:", directory)
        return
    try:
        from IPython.display import Image, display
    except ImportError:
        for path in paths:
            print(path)
        return
    for path in paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))

show_results("outputs/07")


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
